In [1]:
import os
from odps import ODPS
import numpy as np
import pandas as pd

##initialize odps
o = ODPS(
    # （推荐）确保已设置环境变量。
    # 确保ALIBABA_CLOUD_ACCESS_KEY_ID环境变量设置为用户 Access Key ID。
    access_id=os.getenv('ALIBABA_CLOUD_ACCESS_KEY_ID'),
    
    # 确保ALIBABA_CLOUD_ACCESS_KEY_SECRET环境变量设置为用户Access Key Secret。
    secret_access_key=os.getenv('ALIBABA_CLOUD_ACCESS_KEY_SECRET'),
    project='xyf_jingying_dev',
    endpoint='https://service.cn-beijing.maxcompute.aliyun.com/api',
)

import shutil
from openpyxl import load_workbook
from openpyxl.styles import Font, Fill, Border, Alignment, Side, PatternFill
from win32com.client import Dispatch

In [2]:
## 更新数据表
query0='''

'''

## 新客A/B测试分组数据
query1='''
WITH
-- 新客A/B测试分组数据, 按user_no去重，取每个用户的第一次点击时间 
 first_click_ever AS
(
    SELECT  user_no
           ,group_tag
           ,MIN(exposure_time) AS first_exposure_time -- 用户的第一次点击时间 
           ,MIN(exposure_date) AS first_exposure_date -- 用户的第一次点击日期 
    FROM
    (
        SELECT  bizid                                                       AS user_no
               ,CASE WHEN TRIM(group) = 'groupA' THEN '不优化' --分组字符串有空格 
                     WHEN TRIM(group) = 'groupB' THEN '展示优化'  ELSE '其他' END AS group_tag
               ,TO_DATE(datecreated ,'yyyy-mm-dd hh:mi:ss')                 AS exposure_time -- 曝光时间 
               ,TO_DATE(datecreated)                                        AS exposure_date -- 曝光日期 
        FROM xyf_dwd.dwd_msg_biz_random_result_df
        WHERE pt = MAX_PT('xyf_dwd.dwd_msg_biz_random_result_df')
        AND bizkey IN ('cap36xkopt') -- 新客的key 
        -- AND TO_DATE(datecreated) >= '2025-07-29' 
        AND TO_DATE(datecreated) >= '2025-09-02' 
    )
    GROUP BY  user_no
             ,group_tag
), 
-- 授信数据 
credit_info AS (
       SELECT  user_no
              ,cust_no
              ,init_credit_line / 100 AS 授信金额 --单位：分 
              ,created_time           AS 授信时间
              ,credit_expire_date     AS 授信到期时间
       FROM xyf_dwd.dwd_preloan_credit_apply_df
       WHERE pt = MAX_PT('xyf_dwd.dwd_preloan_credit_apply_df')
       AND app IN ('xyf01')
       AND status = 2 --成功 
       AND NVL(app_activation_type, '') <> 'loan_recredit_activation' 
) ,
-- 所有放款订单数据（包含发起和成功）
loan_orders AS (
       SELECT  a.first_order_number
              ,a.first_order_time
              ,a.user_no
              ,a.cust_no
              ,a.period
              ,a.loan_amt
              ,a.loan_time
              ,a.loan_status
              ,a.inner_app
              ,a.app
              ,a.loan_flag
       FROM xyf_dws.dws_inloan_user_order_df a
       WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
       AND a.app = 'xyf01'
       AND inner_app = app  --目前为APP首贷，看是否需要调整
       AND a.loan_flag = '首贷' -- 限制为首贷 
       AND DATE(a.first_order_time) >= '2025-07-29' -- 首单时间范围 
 ) , 
-- 关联分组、授信和放款数据，计算时间差 
user_conversion AS (
       SELECT  fc.user_no
              ,fc.group_tag
              ,fc.first_exposure_time
              ,fc.first_exposure_date
              ,ci.授信金额
       -- 授信金额分层
              ,CASE WHEN ci.授信金额 < 2000 THEN '1.[0,2k)'
                    WHEN ci.授信金额 < 5000 THEN '2.[2k,5k)'
                    WHEN ci.授信金额 < 10000 THEN '3.[5k,10k)'
                    WHEN ci.授信金额 < 20000 THEN '4.[10k,20k)'
                    WHEN ci.授信金额 < 40000 THEN '5.[20k,40k)'
                    WHEN ci.授信金额 >= 40000 THEN '6.[40k+]'
                    ELSE 'unknown'
               END                                   AS 授信金额_level
              ,lo.first_order_number
              ,lo.first_order_time
              ,lo.loan_status
              ,lo.loan_amt
              ,lo.loan_time                       
       FROM first_click_ever fc
       -- 关联授信信息 
       LEFT JOIN credit_info ci
       ON fc.user_no = ci.user_no AND ci.授信时间 < fc.first_exposure_time AND fc.first_exposure_time < ci.授信到期时间
       -- 关联放款信息 
       LEFT JOIN loan_orders lo
       ON fc.user_no = lo.user_no AND lo.first_order_time >= fc.first_exposure_time 
)

-- 计算发起率和通过率统计 
SELECT  group_tag
       ,授信金额_level
       
-- 基础统计 
       ,COUNT(DISTINCT user_no) AS 总曝光用户数

-- 发起率统计（所有订单）
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(first_order_time, first_exposure_time) = 0 THEN user_no END) AS 发起人数_t0
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(first_order_time, first_exposure_time) BETWEEN 0 AND 3 THEN user_no END) AS 发起人数_t3
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(first_order_time, first_exposure_time) BETWEEN 0 AND 7 THEN user_no END) AS 发起人数_t7
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(first_order_time, first_exposure_time) >= 0 THEN user_no END) AS 发起人数_tilnow
       
-- 发起次数统计
       ,COUNT(CASE WHEN DATEDIFF(first_order_time, first_exposure_time) = 0 THEN user_no END) AS 发起次数_t0
       ,COUNT(CASE WHEN DATEDIFF(first_order_time, first_exposure_time) BETWEEN 0 AND 3 THEN user_no END) AS 发起次数_t3
       ,COUNT(CASE WHEN DATEDIFF(first_order_time, first_exposure_time) BETWEEN 0 AND 7 THEN user_no END) AS 发起次数_t7
       ,COUNT(CASE WHEN DATEDIFF(first_order_time, first_exposure_time) >= 0 THEN user_no END) AS 发起次数_tilnow
       
-- 成功放款统计
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(loan_time, first_exposure_time) = 0 THEN user_no END) AS 放款人数_t0
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(loan_time, first_exposure_time) BETWEEN 0 AND 3 THEN user_no END) AS 放款人数_t3
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(loan_time, first_exposure_time) BETWEEN 0 AND 7 THEN user_no END) AS 放款人数_t7
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(loan_time, first_exposure_time) >= 0 THEN user_no END) AS 放款人数_tilnow
       
-- 成功放款次数统计
       ,COUNT(CASE WHEN DATEDIFF(loan_time, first_exposure_time) = 0 THEN user_no END) AS 放款次数_t0
       ,COUNT(CASE WHEN DATEDIFF(loan_time, first_exposure_time) BETWEEN 0 AND 3 THEN user_no END) AS 放款次数_t3
       ,COUNT(CASE WHEN DATEDIFF(loan_time, first_exposure_time) BETWEEN 0 AND 7 THEN user_no END) AS 放款次数_t7
       ,COUNT(CASE WHEN DATEDIFF(loan_time, first_exposure_time) >= 0 THEN user_no END) AS 放款次数_tilnow
              
-- 成功放款金额统计（只统计成功订单）
       ,SUM(CASE WHEN DATEDIFF(loan_time, first_exposure_time) = 0 THEN loan_amt ELSE 0 END) AS 放款金额_t0
       ,SUM(CASE WHEN DATEDIFF(loan_time, first_exposure_time) BETWEEN 0 AND 3 THEN loan_amt ELSE 0 END) AS 放款金额_t3
       ,SUM(CASE WHEN DATEDIFF(loan_time, first_exposure_time) BETWEEN 0 AND 7 THEN loan_amt ELSE 0 END) AS 放款金额_t7
       ,SUM(CASE WHEN DATEDIFF(loan_time, first_exposure_time) >= 0 THEN loan_amt ELSE 0 END) AS 放款金额_tilnow
            
FROM user_conversion
GROUP BY  授信金额_level 
         ,group_tag
ORDER BY  授信金额_level 
         ,group_tag
LIMIT 10000
'''

# 老客A/B测试分组数据
query2='''
SELECT  base_dt
       ,额度区间
       ,group_tag
       ,COUNT(DISTINCT base_user_no )                                                                       AS 客群人数
       ,COUNT(DISTINCT mg_user_no)                                                                          AS 曝光人数
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(create_date,exposure_time) = 0 THEN base_user_no END)             AS 提现人数_t0
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(create_date,exposure_time) BETWEEN 0 AND 3 THEN base_user_no END) AS 提现人数_t3
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(create_date,exposure_time) BETWEEN 0 AND 7 THEN base_user_no END) AS 提现人数_t7
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(create_date,exposure_time) >= 0 THEN base_user_no END)            AS 提现人数_tilnow
       ,COUNT(CASE WHEN DATEDIFF(create_date,exposure_time) = 0 THEN base_user_no END)                      AS 提现次数_t0
       ,COUNT(CASE WHEN DATEDIFF(create_date,exposure_time) BETWEEN 0 AND 3 THEN base_user_no END)          AS 提现次数_t3
       ,COUNT(CASE WHEN DATEDIFF(create_date,exposure_time) BETWEEN 0 AND 7 THEN base_user_no END)          AS 提现次数_t7
       ,COUNT(CASE WHEN DATEDIFF(create_date,exposure_time) >= 0 THEN base_user_no END)                     AS 提现次数_tilnow
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(PAY_date,exposure_time) = 0 THEN base_user_no END)                AS 放款人数_t0
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(PAY_date,exposure_time) BETWEEN 0 AND 3 THEN base_user_no END)    AS 放款人数_t3
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(PAY_date,exposure_time) BETWEEN 0 AND 7 THEN base_user_no END)    AS 放款人数_t7
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(PAY_date,exposure_time) >= 0 THEN base_user_no END)               AS 放款人数_tilnow
       ,COUNT(CASE WHEN DATEDIFF(PAY_date,exposure_time) = 0 THEN base_user_no END)                         AS 放款次数_t0
       ,COUNT(CASE WHEN DATEDIFF(PAY_date,exposure_time) BETWEEN 0 AND 3 THEN base_user_no END)             AS 放款次数_t3
       ,COUNT(CASE WHEN DATEDIFF(PAY_date,exposure_time) BETWEEN 0 AND 7 THEN base_user_no END)             AS 放款次数_t7
       ,COUNT(CASE WHEN DATEDIFF(PAY_date,exposure_time) >= 0 THEN base_user_no END)                        AS 放款次数_tilnow
       ,SUM(CASE WHEN DATEDIFF(PAY_date,exposure_time) = 0 THEN loan_amt ELSE 0 END)                        AS 放款金额_t0
       ,SUM(CASE WHEN DATEDIFF(PAY_date,exposure_time) BETWEEN 0 AND 3 THEN loan_amt ELSE 0 END)            AS 放款金额_t3
       ,SUM(CASE WHEN DATEDIFF(PAY_date,exposure_time) BETWEEN 0 AND 7 THEN loan_amt ELSE 0 END)            AS 放款金额_t7
       ,SUM(CASE WHEN DATEDIFF(PAY_date,exposure_time) >= 0 THEN loan_amt ELSE 0 END)                       AS 放款金额_tilnow
FROM
(
	SELECT  DISTINCT base.dt AS base_dt
	       ,base.user_no     AS base_user_no
	       ,base.cust_no
	       ,base.额度区间
	       ,mg.user_no       AS mg_user_no
	       ,mg.group_tag --分组标签
	       ,mg.exposure_time
	       ,lo.create_date
	       ,lo.PAY_date
	       ,lo.risk_pass
	       ,lo.apply_cnt
	       ,lo.loan_cnt
	       ,lo.apply_amt
	       ,lo.loan_amt
	FROM
	(
		SELECT  user_no
		       ,cust_no
		       ,CASE WHEN available_amt < 500 THEN '1.[0,500)'
		             WHEN available_amt < 1000 THEN '2.[500,1000)'
		             WHEN available_amt < 3000 THEN '3.[1000,3000)'
		             WHEN available_amt < 6000 THEN '4.[3000,6000)'
		             WHEN available_amt < 9000 THEN '5.[6000,9000)'
		             WHEN available_amt < 12000 THEN '6.[9000,12000)'
		             WHEN available_amt < 15000 THEN '7.[12000,15000)'
		             WHEN available_amt >= 15000 THEN '8.[15000,15000+)'  ELSE 'unknown'	
			      END AS 额度区间
		       ,DATE(TO_DATE(pt,'yyyymmdd')) dt
		FROM xyf_ads.ads_user_market_portfolio_label_df
		WHERE pt >= '20250728' 
	) base
	LEFT JOIN
	(
		SELECT  bizid                                              AS user_no
		       ,CASE WHEN TRIM(group) = 'groupA' THEN '不优化'               --分组字符串有空格 
		             WHEN TRIM(group) = 'groupB' THEN '展示优化'  ELSE '其他' END  AS group_tag
		       ,TO_DATE(datecreated ,'yyyy-mm-dd hh:mi:ss')                 AS exposure_time -- 曝光时间 
		       ,TO_DATE(datecreated)                                        AS exposure_date -- 曝光日期
		FROM xyf_dwd.dwd_msg_biz_random_result_df
		WHERE pt = MAX_PT('xyf_dwd.dwd_msg_biz_random_result_df')
		AND bizkey IN ('cap36lkopt')   -- 老客的key 
		AND TO_DATE(datecreated) >= '2025-07-29' 
		QUALIFY ROW_NUMBER() OVER ( PARTITION BY bizid 
		                           ORDER BY TO_DATE(datecreated ,'yyyy-mm-dd hh:mi:ss') ASC ) = 1 
	) mg
	ON base.user_no = mg.user_no AND mg.exposure_date = DATE_ADD(base.dt, 1)
	LEFT JOIN
	(
		SELECT  user_no
		       ,first_order_number
		       ,DATE(first_order_time)
		       ,MAX(first_order_time)                                                AS create_date
		       ,MAX(loan_time)                                                       AS PAY_date
		       ,MAX(CASE WHEN risk_status = "pass" THEN 1 ELSE 0 END)                AS risk_pass -- 风控通过 
			,MAX(CASE WHEN first_order_time IS NOT NULL THEN 1 ELSE 0 END)        AS apply_cnt -- 提现人
		       ,MAX(CASE WHEN loan_time IS NOT NULL THEN 1 ELSE 0 END)               AS loan_cnt -- 放款人 
		       ,MAX(CASE WHEN first_order_time IS NOT NULL THEN loan_amt ELSE 0 END) AS apply_amt -- 提现金额 
		       ,SUM(CASE WHEN loan_time IS NOT NULL THEN loan_amt ELSE 0 END)        AS loan_amt --放款金额
			-- , MAX(CASE WHEN loan_status = "success" AND loan_time IS NOT NULL THEN period ELSE 0 END) AS apply_period
		FROM xyf_dws.dws_inloan_user_order_df
		WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
		AND app IN ('xyf01') --看情况要不要'fxk' 
		AND inner_app = app
		AND DATE(first_order_time) >= '2025-07-29'
		AND loan_flag <> '首贷'
		GROUP BY  user_no
		         ,DATE(first_order_time)
		         ,first_order_number
	) lo
	ON mg.user_no = lo.user_no AND DATEDIFF(create_date, exposure_time) BETWEEN 0 AND 30 --限制t30内发起，根据实际情况修改
	WHERE base.dt >= '2025-07-29'
	AND mg.user_no IS NOT NULL 
)
GROUP BY  base_dt
         ,额度区间
         ,group_tag
'''


In [3]:
## 更新相关参数

#新客数据
# result = o.execute_sql(query0)
result = o.execute_sql(query1)
xinke = result.open_reader().to_pandas()

#老客数据
result = o.execute_sql(query2)
laoke = result.open_reader().to_pandas()

# ===========================================
# 1.新客  
# ===========================================
# 日期和字符串字段
# xinke['订单发起日期'] = pd.to_datetime(xinke['订单发起日期']).dt.strftime('%Y-%m-%d')
xinke['group_tag'] = xinke['group_tag'].astype(str)
xinke['授信金额_level'] = xinke['授信金额_level'].astype(str)

# # 浮点数字段 - 保留2位小数
# f_columns_asset = []
# for col in f_columns_asset:
#     if col in xinke.columns:
#         xinke[col] = xinke[col].astype('float64')

# 整数字段
int_columns_asset = [ '总曝光用户数',
                      '发起人数_t0', '发起人数_t3', '发起人数_t7', '发起人数_tilnow',
                      '发起次数_t0', '发起次数_t3', '发起次数_t7', '发起次数_tilnow',
                      '放款人数_t0', '放款人数_t3', '放款人数_t7', '放款人数_tilnow',
                      '放款次数_t0', '放款次数_t3', '放款次数_t7', '放款次数_tilnow',
                      '放款金额_t0', '放款金额_t3', '放款金额_t7', '放款金额_tilnow']
for col in int_columns_asset:
    if col in xinke.columns:
        xinke[col] = xinke[col].astype('int64')


# ===========================================
# 2. 老客
# ===========================================
# 日期和字符串字段
laoke['base_dt'] = pd.to_datetime(laoke['base_dt']).dt.strftime('%Y-%m-%d')
laoke['额度区间'] = laoke['额度区间'].astype(str)
laoke['group_tag'] = laoke['group_tag'].astype(str)

# # 浮点数字段 - 保留2位小数
# f_columns_loan = []
# for col in f_columns_loan:
#     if col in laoke.columns:
#         laoke[col] = laoke[col].astype('float64')

# 整数字段
int_columns_loan = ['客群人数', '曝光人数', 
                    '提现人数_t0', '提现人数_t3', '提现人数_t7', '提现人数_tilnow',
                    '提现次数_t0', '提现次数_t3', '提现次数_t7', '提现次数_tilnow',
                    '放款人数_t0', '放款人数_t3', '放款人数_t7', '放款人数_tilnow',
                    '放款次数_t0', '放款次数_t3', '放款次数_t7', '放款次数_tilnow',
                    '放款金额_t0', '放款金额_t3', '放款金额_t7', '放款金额_tilnow']
for col in int_columns_loan:
    if col in laoke.columns:
        laoke[col] = laoke[col].astype('int64')



In [ ]:
def write_dataframe_to_excel_com(file_path, dataframes_dict):
    """
    Args:
        file_path: Excel文件路径
        dataframes_dict: {sheet_name: dataframe} 字典
    """
    # 启动Excel应用
    excel_app = Dispatch("Excel.Application")
    excel_app.Visible = False  # 后台运行
    excel_app.DisplayAlerts = False  # 禁用警告对话框
    
    try:
        # 打开工作簿
        workbook = excel_app.Workbooks.Open(file_path)
        
        for sheet_name, df in dataframes_dict.items():
            try:
                # 获取工作表
                worksheet = workbook.Worksheets(sheet_name)
                
                # 清空从第2行开始的旧数据（保留第1行表头）
                if worksheet.UsedRange.Rows.Count > 1:
                    # 计算要清空的范围：从第2行到最后一行
                    last_row = worksheet.UsedRange.Rows.Count
                    last_col = worksheet.UsedRange.Columns.Count
                    if last_row > 1:
                        clear_range = worksheet.Range(
                            worksheet.Cells(2, 1), 
                            worksheet.Cells(last_row, last_col)
                        )
                        clear_range.ClearContents()
                
                # 将DataFrame转换为列表（从第2行开始写入）
                if not df.empty:
                    # 获取数据范围
                    start_row = 2  # 从第2行开始
                    start_col = 1  # 从第1列开始
                    end_row = start_row + len(df) - 1
                    end_col = start_col + len(df.columns) - 1
                    
                    # 转换DataFrame为二维列表
                    data_list = df.values.tolist()
                    
                    # 定义写入范围
                    write_range = worksheet.Range(
                        worksheet.Cells(start_row, start_col),
                        worksheet.Cells(end_row, end_col)
                    )
                    
                    # 写入数据
                    write_range.Value = data_list
                    
                print(f"成功写入工作表: {sheet_name}")
                
            except Exception as e:
                print(f"写入工作表 {sheet_name} 时出错: {e}")
                continue
        
        # 刷新所有数据连接和透视表
        workbook.RefreshAll()
        # 强制重新计算公式
        excel_app.Calculate()
        # 等待所有操作完成
        excel_app.CalculateUntilAsyncQueriesDone()

        # 保存并关闭
        workbook.Save()
        workbook.Close()
        print(f"文件已保存: {file_path}")
        
    except Exception as e:
        print(f"操作Excel文件时出错: {e}")
        
    finally:
        # 退出Excel应用
        excel_app.Quit()

# 使用示例 - 替换你的pd.ExcelWriter代码
file_path = 'D:/cap36利益点提前结清测试/cap36展示优化（新老客).xlsx'

dataframes_dict = {
    '新客': xinke,
    '老客': laoke
}

write_dataframe_to_excel_com(file_path, dataframes_dict)

成功写入工作表: 新客
成功写入工作表: 老客
文件已保存: D:/cap36利益点提前结清测试/cap36展示优化（新老客).xlsx
